# Aashish AI — Stage B training, instruction tuning (§7.4, P5)

Continues a Stage A checkpoint on the §7.4 instruction data with **assistant-only
loss**. It exists because Stage B had data and no trainer, and that mattered more
than it sounds: the runtime frame is essentially absent from the pretraining
corpus — of 17,265 seed documents, `<|ctx|>` appears in **4**, `<|asst|>` in
1,108, `<|sys|>` in 415. A Stage A-only checkpoint has therefore never been
shown the format it is asked to continue at inference time, and no amount of
extra pretraining steps on the same shards changes that.

**Status: the path is MEASURED at the local shape, the scale is NOT TESTED.**
`training/checkpoints/sft-local` holds a 60-step run from this laptop: loss
7.5571 → 6.3133 (windowed gate PASS), val 6.2715 → 5.5403, and the same prompts
through `inference/sample_answers.py` show the frame and `<|end|>` appearing
where Stage A emitted neither. Sixty steps on 3,000 examples is a wiring proof,
not quality. **No Kaggle session has run this notebook yet**, so treat the first
run as the test.

Read section 4 before spending GPU hours: the mask is the whole contribution of
Stage B, and it is printed rather than asserted.

## 1. Setup

In [ ]:
!nvidia-smi

import subprocess
import sys

import torch

print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
print('python', sys.version.split()[0], '| commit',
      subprocess.run(['git', 'rev-parse', '--short', 'HEAD'],
                     capture_output=True, text=True).stdout.strip())
print('bf16 supported:',
      torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)
# A T4 has no bf16 — which is why this notebook uses fp16 AMP, like Stage A.

In [ ]:
import os
import pathlib
import shutil

CANDIDATES = [
    '/kaggle/working/aashish-ai-portfolio',
    '/kaggle/working/newportfolio',
    '/kaggle/input/aashish-ai-portfolio',
]
for candidate in CANDIDATES:
    if pathlib.Path(candidate, 'training').is_dir():
        os.chdir(candidate)
        break
else:
    raise SystemExit(f'Set CANDIDATES to the repository location; looked in {CANDIDATES}')

# /kaggle/input is a read-only mount, and this run writes data/instruction/,
# the tokenizer artifact and the checkpoints. A Dataset has to be copied to
# /kaggle/working first, and a working copy has to win over the mount, or the
# first write dies with OSError: [Errno 30]. Fail here instead of later.
if pathlib.Path.cwd().is_relative_to('/kaggle/input'):
    target = pathlib.Path('/kaggle/working/aashish-ai-portfolio')
    if not target.exists():
        shutil.copytree(pathlib.Path.cwd(), target)
    os.chdir(target)
pathlib.Path('.tmp-writable').write_text('ok')
pathlib.Path('.tmp-writable').unlink()

TOKENIZER = 'ai/tokenizer/artifacts/stage-a-12k'
DATA = 'data/instruction/sft.jsonl'
STAGE_A = '/kaggle/working/checkpoints/stage-a'
RUN = '/kaggle/working/checkpoints/stage-b'

print('cwd:', os.getcwd())
print('tokenizer:', pathlib.Path(TOKENIZER, 'tokenizer.json').is_file())
print('instruction data:', pathlib.Path(DATA).is_file())
print('stage A checkpoint:', pathlib.Path(STAGE_A, 'latest.pt').is_file())

# `python -m` from here puts the repo root on sys.path, which is why every
# command below is a module rather than a path.

## 2. What Stage A left behind

Stage B continues a Stage A checkpoint. `--init` loads **model weights only**:
the optimizer, scaler and scheduler start fresh, because Stage B is a different
objective on different data and carrying Adam moments across is not "resuming".
The flag refuses a checkpoint whose vocab, width or depth differ from this run's
config — a checkpoint from another shape would otherwise load some keys and
leave the rest at their initial values, with no error to notice.

In [ ]:
!ls -la $STAGE_A

# Read the step count and the loss history here. Stage B's run manifest records
# what it was initialised from, so the two runs stay joined up on the record
# rather than in memory.
import json

manifest = json.load(open(f'{STAGE_A}/RUN_MANIFEST.json', encoding='utf-8'))
print('stage A step', manifest['hyperparameters']['steps'],
      '| tokenizer', manifest['tokenizer_version'],
      '| params', f"{manifest['params']:,}")
print('tokenizer must match this notebook:', manifest['tokenizer_version'] ==
      json.load(open(f'{TOKENIZER}/meta.json', encoding='utf-8'))['tokenizer_version'])

## 3. The instruction data (§7.4)

40,000 examples generated from `knowledge.json` — context-grounded, ≥30%
counterfactual contexts, 15% abstention, the §7.4 mix exact by largest-remainder
allocation. Nothing private is in it: the generator's fact index is the public
view only, because a model that learns a withheld value is one prompt away from
emitting it and no runtime filter can un-learn it.

In [ ]:
!python -m training.scripts.make_instruction_data --count 40000

# The run prints the mix it achieved against the mix it asked for, the
# counterfactual share, and — now that the tokenizer exists — the token count
# MEASURED with the shipping tokenizer. Do not schedule from the ESTIMATED
# figure: on the 40k set it was 7,263,195 against a measured 10,582,527, i.e.
# 31% low, because Hindi and Hinglish tokenize near 2.3 characters per token
# rather than the 3.4 the estimate assumed.
!cat data/instruction/manifest.json

## 4. Look at the mask before spending GPU hours

Stage B's entire contribution is *which tokens the loss is on*. A wrong mask
does not crash and does not bend the loss curve in an obviously wrong way: the
run trains on the question, or silently drops the first word of every answer,
and the result is a model that writes plausible questions instead of answers.

This pass needs no torch and prints the mask for real examples.

In [ ]:
!python -m training.scripts.train_stage_b --pipeline-only --tokenizer $TOKENIZER --data $DATA --block 1024 --batch 8

# Read the `loss is on:` lines. They must be the assistant's answers plus their
# `<|end|>` tokens — no question, no rules, no `<|sys|>`/`<|ctx|>`/`<|user|>`.
# The pass also prints the supervised share of the stream (about 10.9%), which
# is the number to size a step budget from: an epoch is 10.58M tokens, of which
# only ~1.16M carry loss.
#
# It also checks the two things that would otherwise be found by a NaN or a
# silent no-op: every window handed to the model carries supervision (an
# all-ignored target makes cross_entropy return NaN and one NaN step poisons
# every weight in the run), and a saved cursor regenerates the identical next
# batch.

## 5. Persistence first

Kaggle sessions are ephemeral. Push or save the run directory before training,
not after — a session that ends without either loses everything.

In [ ]:
#  - push /kaggle/working/checkpoints/stage-b to a Kaggle Dataset / Drive / HF Hub, or
#  - write to /kaggle/working and Save Version every session.
!mkdir -p $RUN
!ls -la $RUN

## 6. Stage B training

`--max-minutes` stops cleanly and writes a checkpoint before the time box ends,
so an interrupted run is a resume rather than a loss. The learning rate is
lower than Stage A's on purpose: starting an SFT run at the pretraining rate
undoes the pretraining.

In [ ]:
!python -m training.scripts.train_stage_b --config A --tokenizer $TOKENIZER --data $DATA --init $STAGE_A/latest.pt --run-dir $RUN --steps 3000 --batch 8 --block 1024 --grad-accum 2 --lr 5e-5 --warmup 100 --eval-every 100 --save-every 100 --amp --gate --max-minutes 540 --resume auto

# 8 x 1024 x 2 is ~16K tokens per optimizer step, of which roughly 10.9% carry
# loss. The gate is the windowed one: first five logged steps against the last
# last five, so a run that ends slightly above its minimum cannot fail it.

## 7. Every subsequent session

The identical command. It resumes from `latest.pt` and reports the step, the
tokens and supervised tokens already consumed, and the loss history.

In [ ]:
!ls -la $RUN
!cat $RUN/RUN_MANIFEST.json

# A resume that silently began a new run would look like success here, so check
# that the step continues from where the previous session stopped, and that the
# `tokens_seen_cumulative` figure is not zero. A zeroed cursor reads exactly
# like a lost data cursor — that bug existed and was found by running this.

## 8. See the answers, not just the loss

A low loss and a useful answer are different claims. This samples the runtime
frame — the prompt the browser builds, cut at `<|asst|>` — through the same
numpy forward the JS engine is verified against, and prints both checkpoints
side by side. It reports the prompt echo rate too, because an answer that
copies its input scores well and is useless.

In [ ]:
!python inference/sample_answers.py --checkpoint stage-A=$STAGE_A --checkpoint stage-B=$RUN --n 4 --tokens 32

# What to look for, in order:
#   1. does the answer end with `<|end|>` (does the model know a turn ends)?
#   2. does it answer, or does it echo the context back?
#   3. does it abstain with `<|abstain|>` when the context has no answer?
# A run that has learnt (1) but not (2) is a run that needs more of the same
# data, not different hyperparameters.

## 9. Export (P6)

`npm run export:model -- --run-dir <this run>` then `npm run verify:engine`,
which is the gate that compares the JavaScript engine against the numpy
reference **on the shipping weights**. Export is not scaffolded further here on
purpose: it needs a finished checkpoint, and a pipeline built against a
checkpoint that does not exist yet is a pipeline whose interface is invented.